# Zero-shot vs Few-shot Comparison

## Zero-shot vs Few-shot Learning in LLMs

Both terms describe how you give an LLM instructions/examples in a prompt to get it to perform a task — without updating the model's weights (no fine-tuning involved).

## Zero-shot

You give the model **only an instruction**, with no examples of what the output should look like. The model relies entirely on what it learned during pretraining to figure out the task.

**Example:**
```
Classify the sentiment of this review as Positive, Negative, or Neutral.

Review: "The food was cold and the service was painfully slow."
Sentiment:
```
The model has never seen a demonstration in this prompt — it just infers from the instruction and its general knowledge that this is a sentiment classification task, and answers: `Negative`.

## Few-shot

You give the model **a handful of examples** (input → output pairs) before asking it to do the same task on a new input. This helps the model understand the exact format, style, or edge-case handling you want, especially for tasks that are ambiguous or unusual.

**Example:**
```
Classify the sentiment of these reviews as Positive, Negative, or Neutral.

Review: "Absolutely loved the ambiance and the staff!"
Sentiment: Positive

Review: "It was okay, nothing special."
Sentiment: Neutral

Review: "Waited an hour and the order was still wrong."
Sentiment: Negative

Review: "The food was cold and the service was painfully slow."
Sentiment:
```
By seeing 3 examples first (this is "3-shot"), the model locks onto the exact label vocabulary (`Positive`/`Negative`/`Neutral`), the format (`Sentiment: <label>`), and the calibration of what counts as "Neutral" vs "Negative" — often improving accuracy over zero-shot, especially for niche or precisely-defined tasks.

## Key differences

| | Zero-shot | Few-shot |
|---|---|---|
| Examples given | 0 | Usually 1–10+ ("one-shot" = 1, "few-shot" = 2+) |
| Relies on | Pretrained general knowledge | Pattern-matching from in-prompt examples |
| Best for | Simple, well-known tasks | Ambiguous formats, unusual labels, style matching |
| Prompt length | Short | Longer (more tokens used) |
| Risk | Model may misinterpret task intent | Model may overfit to quirks of the examples given |

**A quick analogy:** zero-shot is like asking a new employee to "write a polite decline email" and trusting their judgment. Few-shot is like showing them 3 sample decline emails your company has sent before, then asking them to write a new one in the same style.

Both are distinct from **fine-tuning**, where you actually update the model's weights using a training dataset — few-shot prompting achieves something similar "on the fly," purely through context, without any retraining.

## When to use which

**Use Zero-shot when:**
- The task is simple, common, or well-known (e.g., translation, summarization, basic sentiment analysis).
- You want to save tokens/cost and keep prompts short.
- The model already performs well without examples (test it first — modern LLMs are quite good zero-shot).

**Use Few-shot when:**
- The task has a specific/unusual output format you need followed exactly (e.g., custom JSON schema, specific labels like "Urgent/Routine/Spam").
- The task is ambiguous and examples remove guesswork (e.g., "what counts as sarcasm" for your use case).
- You need consistent style/tone matching (e.g., matching a brand's email voice).
- Zero-shot attempts gave wrong or inconsistent results — few-shot is your next fix before considering fine-tuning.

## Simple rule of thumb

1. **Start with zero-shot** — it's cheaper and faster.
2. **If output quality/format is off**, add 2–5 well-chosen few-shot examples covering edge cases.
3. **If even few-shot isn't reliable enough** at scale, consider fine-tuning instead.

**One-line heuristic:** *Zero-shot for general tasks the model already "knows"; few-shot for tasks needing a specific format, style, or precision that only examples can convey.*

Here are practical code examples using the OpenAI-style API (the same pattern works for Claude, Gemini, etc. — just the client differs).

## Zero-shot Example (Sentiment Classification)

In [1]:
from openai import OpenAI

client = OpenAI()

prompt = """Classify the sentiment of this review as Positive, Negative, or Neutral.

Review: "The food was cold and the service was painfully slow."
Sentiment:"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)
# Output: Negative

Negative


## Few-shot Example (Sentiment Classification)


In [2]:
from openai import OpenAI

client = OpenAI()

prompt = """Classify the sentiment of these reviews as Positive, Negative, or Neutral.

Review: "Absolutely loved the ambiance and the staff!"
Sentiment: Positive

Review: "It was okay, nothing special."
Sentiment: Neutral

Review: "Waited an hour and the order was still wrong."
Sentiment: Negative

Review: "The food was cold and the service was painfully slow."
Sentiment:"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)
# Output: Negative

Negative


## Few-shot Using the `messages` Structure (Alternative, Cleaner Approach)

Instead of cramming examples into one string, you can use the chat `messages` array itself to simulate a conversation of examples:


In [3]:
from openai import OpenAI

client = OpenAI()

messages = [
    {"role": "system", "content": "Classify sentiment as Positive, Negative, or Neutral."},
    {"role": "user", "content": "Absolutely loved the ambiance and the staff!"},
    {"role": "assistant", "content": "Positive"},
    {"role": "user", "content": "It was okay, nothing special."},
    {"role": "assistant", "content": "Neutral"},
    {"role": "user", "content": "Waited an hour and the order was still wrong."},
    {"role": "assistant", "content": "Negative"},
    {"role": "user", "content": "The food was cold and the service was painfully slow."}
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages
)

print(response.choices[0].message.content)
# Output: Negative

Negative


This approach mimics real dialogue turns, which some models handle even more reliably than a single long prompt string — useful when you want the "examples" to feel like natural prior exchanges rather than a static block of text.

**Key takeaway in code terms:** zero-shot = 1 instruction + 1 query in the prompt; few-shot = instruction + N example pairs + query, either concatenated in one string or spread across `messages`.